#### 01. CARREGANDO MAPEAMENTO DE PASTAS E IMPORTS

In [1]:

# Importando bibliotecas
from functions import *
import pandas as pd
import locale
from pathlib import Path
import shutil
from datetime import datetime
import warnings
import logging
from openpyxl import load_workbook

timer = Temporizador()

timer.iniciar()

locale.setlocale(locale.LC_TIME, 'Portuguese_Brazil.1252')  # Para Windows
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.expand_frame_repr', False)

# Detecta se o script está sendo executado de um .py ou de um notebook
try:
    caminho_base = Path(__file__).resolve().parent
except NameError:
    # __file__ não existe em Jupyter ou ambiente interativo
    caminho_base = Path.cwd()

pasta_input_parquet = caminho_base.parent / '01_INPUT_PIPELINE/01_BD_PARQUET'
pasta_staging_parquet = caminho_base.parent / '02_STAGING_PARQUET'
pasta_painel = caminho_base.parent / '05_PAINEL'
pasta_historico_planos = caminho_base.parent / '04_HISTORICO_PLANOS'
pasta_saida_plan_producao = caminho_base.parent / '06_SAIDA_PLAN_PRODUCAO'

print("✅ Mapeamento de pastas concluído com sucesso!")

✅ Mapeamento de pastas concluído com sucesso!


#### 02. CARREGANDO PLANO REGIONAL DO EXCEL

In [64]:
# -----------------------------------------------------------------------
# Carregar plano do painel em Excel do ciclo atual - REGIONAL
# -----------------------------------------------------------------------

# Definir arquivo original do painel e arquivo temporário
origem = pasta_painel / 'PREV_DEMANDA_KRONA.xlsb'
copia = pasta_painel / 'PREV_DEMANDA_KRONA_TEMP.xlsb'

# Excluir arquivo temporário anterior, caso exista
if copia.exists():
    copia.unlink()

# Criar uma cópia temporária do arquivo original
shutil.copy2(origem, copia)

# Carregar a aba do plano regional a partir da 4ª linha
df_plano_regional_excel = pd.read_excel(copia, sheet_name='PREV_COLAB_REGIONAL', engine='pyxlsb', skiprows=5)

# Definir os rótulos que devem ser mantidos
colunas_manter = ['REGIONAL', 'REGIONAL GESTOR', 'FAMILIA', 'ORÇAMENTO [KG]', 'ORÇAMENTO [R$]', 'ESTATÍSTICO [KG]', 'TOTAL CONSENSO [KG]', 'CONSENSO [R$]']

# Ler e normalizar os rótulos da segunda linha do DataFrame
rotulos = df_plano_regional_excel.iloc[1].astype(str).str.replace('\n', ' ', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip().str.upper()

# Normalizar os nomes das colunas desejadas para comparação
colunas_manter_norm = [coluna.upper() for coluna in colunas_manter]

# Identificar as colunas físicas correspondentes aos rótulos desejados
colunas_selecionadas = [coluna for coluna, rotulo in zip(df_plano_regional_excel.columns, rotulos) if rotulo in colunas_manter_norm]

# Manter somente as colunas selecionadas
df_plano_regional_excel = df_plano_regional_excel[colunas_selecionadas]

# Eliminar linha 0 e reorganizar o índice
df_plano_regional_excel = df_plano_regional_excel.drop(index=0).reset_index(drop=True)

# Ler e normalizar os rótulos reais da primeira linha do DataFrame
rotulos = df_plano_regional_excel.iloc[0].astype(str).str.replace('\n', ' ', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip()

# Identificar as colunas fixas
colunas_id = [coluna for coluna, rotulo in zip(df_plano_regional_excel.columns, rotulos) if rotulo in ['REGIONAL GESTOR', 'REGIONAL', 'FAMILIA']]

# Identificar as colunas de período
colunas_periodo = [coluna for coluna in df_plano_regional_excel.columns if coluna not in colunas_id]

# Criar mapa entre coluna física e métrica correspondente
map_metrica = {coluna: rotulo for coluna, rotulo in zip(df_plano_regional_excel.columns, rotulos) if coluna in colunas_periodo}

# Remover a linha de rótulos antes da transposição
df_plano_regional_excel = df_plano_regional_excel.iloc[1:].reset_index(drop=True)

# Transpor as colunas mensais para linhas
df_plano_regional_excel = df_plano_regional_excel.melt(id_vars=colunas_id, value_vars=colunas_periodo, var_name='PERIODO', value_name='VALOR')

# Criar coluna com a métrica correspondente
df_plano_regional_excel['METRICA'] = df_plano_regional_excel['PERIODO'].map(map_metrica)

# Remover os sufixos .1, .2, .3 etc. gerados pelo pandas nas colunas de período
df_plano_regional_excel['PERIODO'] = df_plano_regional_excel['PERIODO'].astype(str).str.replace(r'\.\d+$', '', regex=True)

# Converter o período serial do Excel para data
df_plano_regional_excel['PERIODO'] = pd.to_datetime(pd.to_numeric(df_plano_regional_excel['PERIODO'], errors='coerce'), unit='D', origin='1899-12-30')

# Renomear as colunas fixas para o padrão final
map_colunas_id = {coluna: ('REGIONAL_GESTOR' if rotulo == 'REGIONAL GESTOR' else rotulo) for coluna, rotulo in zip(df_plano_regional_excel.columns[:len(colunas_id)], ['REGIONAL GESTOR', 'REGIONAL', 'FAMILIA'])}
df_plano_regional_excel = df_plano_regional_excel.rename(columns=map_colunas_id)

# Organizar as colunas finais
df_plano_regional_excel = df_plano_regional_excel[['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'PERIODO', 'METRICA', 'VALOR']]

# Padronizar os nomes das métricas
df_plano_regional_excel['METRICA'] = df_plano_regional_excel['METRICA'].replace({
    'Estatístico [KG]': 'ESTATISTICO_KG',
    'Orçamento [KG]': 'ORCAMENTO_KG',
    'Orçamento [R$]': 'ORCAMENTO_VAL',
    'Total Consenso [KG]': 'CONSENSO_KG',
    'Consenso [R$]': 'CONSENSO_VAL'
})

# Identificar períodos sem planejamento com base no total de CONSENSO_KG
periodos_excluir = df_plano_regional_excel.loc[df_plano_regional_excel['METRICA'].eq('CONSENSO_KG')].groupby('PERIODO')['VALOR'].sum()
periodos_excluir = periodos_excluir[periodos_excluir <= 0].index

# Excluir todas as linhas dos períodos sem planejamento
df_plano_regional_excel = df_plano_regional_excel.loc[~df_plano_regional_excel['PERIODO'].isin(periodos_excluir)].copy()

# Garantir VALOR como numérico antes do pivot
df_plano_regional_excel['VALOR'] = pd.to_numeric(df_plano_regional_excel['VALOR'], errors='coerce').fillna(0)

# Transformar os valores da coluna METRICA em colunas
df_plano_regional_excel = df_plano_regional_excel.pivot_table(index=['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'PERIODO'], columns='METRICA', values='VALOR', aggfunc='sum', fill_value=0).reset_index()

# Remover o nome do eixo criado pelo pivot
df_plano_regional_excel.columns.name = None

# Criar colunas adicionais para produto e grupo de cliente
df_plano_regional_excel[['COD_PROD', 'DESC_PROD', 'COD_GRP_CLIENTE', 'DESC_GRP_CLIENTE']] = ''

# Organizar as colunas na mesma estrutura do Parquet histórico
df_plano_regional_excel = df_plano_regional_excel[['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'COD_PROD', 'DESC_PROD', 'COD_GRP_CLIENTE', 'DESC_GRP_CLIENTE', 'PERIODO', 'ESTATISTICO_KG', 'ORCAMENTO_KG', 'ORCAMENTO_VAL', 'CONSENSO_KG', 'CONSENSO_VAL']]

# Criar coluna PLANO com o valor REGIONAL
df_plano_regional_excel['PLANO'] = 'REGIONAL'

# Criar o CICLO com base no menor e maior PERIODO do plano
meses_abrev = {1: 'JAN', 2: 'FEV', 3: 'MAR', 4: 'ABR', 5: 'MAI', 6: 'JUN', 7: 'JUL', 8: 'AGO', 9: 'SET', 10: 'OUT', 11: 'NOV', 12: 'DEZ'}

periodo_min = df_plano_regional_excel['PERIODO'].min()
periodo_max = df_plano_regional_excel['PERIODO'].max()

ciclo = f"CICLO_{meses_abrev[periodo_min.month]}{str(periodo_min.year)[-2:]}_{meses_abrev[periodo_max.month]}{str(periodo_max.year)[-2:]}"
df_plano_regional_excel['CICLO'] = ciclo

# Criar REVISAO com base na última revisão existente para o mesmo CICLO
df_bd_plano_painel_sop = pd.read_parquet(pasta_historico_planos / 'BD_PLANO_PAINEL_SOP.parquet')

if ciclo in df_bd_plano_painel_sop['CICLO'].values:
    revisoes_ciclo = df_bd_plano_painel_sop.loc[df_bd_plano_painel_sop['CICLO'].eq(ciclo), 'REVISAO']
    ultima_revisao_num = revisoes_ciclo.str.extract(r'(\d+)')[0].astype(int).max()
    nova_revisao = f'REV{ultima_revisao_num + 1}'
else:
    nova_revisao = 'REV1'

df_plano_regional_excel['REVISAO'] = nova_revisao

# Criar coluna DATA_VERSAO com a data e hora atual
df_plano_regional_excel['DATA_VERSAO'] = pd.Timestamp.now()

# Padronizar DATA_VERSAO para datetime64[ns]
df_plano_regional_excel['DATA_VERSAO'] = df_plano_regional_excel['DATA_VERSAO'].astype('datetime64[ns]')

# Padronizar as métricas numéricas como float
colunas_metricas = ['ESTATISTICO_KG', 'ORCAMENTO_KG', 'ORCAMENTO_VAL', 'CONSENSO_KG', 'CONSENSO_VAL']
df_plano_regional_excel[colunas_metricas] = df_plano_regional_excel[colunas_metricas].astype(float)

# Padronizar os códigos de produto e grupo de cliente como strings sem espaços em branco
df_plano_regional_excel['COD_PROD'] = df_plano_regional_excel['COD_PROD'].astype('string').str.strip()
df_plano_regional_excel['COD_GRP_CLIENTE'] = df_plano_regional_excel['COD_GRP_CLIENTE'].astype('string').str.strip()

# Excluir o arquivo temporário, se existir
if copia.exists():
    copia.unlink()

#### 03. CARREGANDO PLANO CLIENTE DO EXCEL

In [65]:
# -----------------------------------------------------------------------
# Carregar plano do painel em Excel do ciclo atual - CLIENTE
# -----------------------------------------------------------------------

# Definir arquivo original do painel e arquivo temporário
origem = pasta_painel / 'PREV_DEMANDA_KRONA.xlsb'
copia = pasta_painel / 'PREV_DEMANDA_KRONA_TEMP.xlsb'

# Excluir arquivo temporário anterior, caso exista
if copia.exists():
    copia.unlink()

# Criar uma cópia temporária do arquivo original
shutil.copy2(origem, copia)

# Carregar a aba do plano a partir da 4ª linha
df_plano_cliente_excel = pd.read_excel(copia, sheet_name='PREV_COLAB_CLIENTE', engine='pyxlsb', skiprows=5)

# Definir os rótulos que devem ser mantidos
colunas_manter = ['REGIONAL', 'REGIONAL GESTOR', 'FAMILIA', 'COD GRP CLIENTE', 'DESC GRP CLIENTE', 'ESTATÍSTICO [KG]', 'CONSENSO [KG]', 'CONSENSO [R$]']

# Ler e normalizar os rótulos da segunda linha do DataFrame
rotulos = df_plano_cliente_excel.iloc[1].astype(str).str.replace('\n', ' ', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip().str.upper()

# Normalizar os nomes das colunas desejadas para comparação
colunas_manter_norm = [coluna.upper() for coluna in colunas_manter]

# Identificar as colunas físicas correspondentes aos rótulos desejados
colunas_selecionadas = [coluna for coluna, rotulo in zip(df_plano_cliente_excel.columns, rotulos) if rotulo in colunas_manter_norm]

# Manter somente as colunas selecionadas
df_plano_cliente_excel = df_plano_cliente_excel[colunas_selecionadas]

# Eliminar linha 0 e reorganizar o índice
df_plano_cliente_excel = df_plano_cliente_excel.drop(index=0).reset_index(drop=True)

# Ler e normalizar os rótulos reais da primeira linha do DataFrame
rotulos = df_plano_cliente_excel.iloc[0].astype(str).str.replace('\n', ' ', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip()

# Identificar as colunas fixas
colunas_id = [coluna for coluna, rotulo in zip(df_plano_cliente_excel.columns, rotulos) if rotulo in ['REGIONAL GESTOR', 'REGIONAL', 'FAMILIA', 'COD GRP CLIENTE', 'DESC GRP CLIENTE']]

# Identificar as colunas de período
colunas_periodo = [coluna for coluna in df_plano_cliente_excel.columns if coluna not in colunas_id]

# Criar mapa entre coluna física e métrica correspondente
map_metrica = {coluna: rotulo for coluna, rotulo in zip(df_plano_cliente_excel.columns, rotulos) if coluna in colunas_periodo}

# Remover a linha de rótulos antes da transposição
df_plano_cliente_excel = df_plano_cliente_excel.iloc[1:].reset_index(drop=True)

# Transpor as colunas mensais para linhas
df_plano_cliente_excel = df_plano_cliente_excel.melt(id_vars=colunas_id, value_vars=colunas_periodo, var_name='PERIODO', value_name='VALOR')

# Criar coluna com a métrica correspondente
df_plano_cliente_excel['METRICA'] = df_plano_cliente_excel['PERIODO'].map(map_metrica)

# Remover os sufixos .1, .2, .3 etc. gerados pelo pandas nas colunas de período
df_plano_cliente_excel['PERIODO'] = df_plano_cliente_excel['PERIODO'].astype(str).str.replace(r'\.\d+$', '', regex=True)

# Converter o período serial do Excel para data
df_plano_cliente_excel['PERIODO'] = pd.to_datetime(pd.to_numeric(df_plano_cliente_excel['PERIODO'], errors='coerce'), unit='D', origin='1899-12-30')

# Renomear as colunas fixas para o padrão final
map_colunas_id = {coluna: ('REGIONAL_GESTOR' if rotulo == 'REGIONAL GESTOR' else ('REGIONAL' if rotulo == 'REGIONAL' else ('FAMILIA' if rotulo == 'FAMILIA' else ('COD_GRP_CLIENTE' if rotulo == 'COD GRP CLIENTE' else ('DESC_GRP_CLIENTE' if rotulo == 'DESC GRP CLIENTE' else rotulo))))) for coluna, rotulo in zip(df_plano_cliente_excel.columns[:len(colunas_id)], ['REGIONAL GESTOR', 'REGIONAL', 'COD GRP CLIENTE', 'DESC GRP CLIENTE', 'FAMILIA'])}
df_plano_cliente_excel = df_plano_cliente_excel.rename(columns=map_colunas_id)

# Organizar as colunas finais
df_plano_cliente_excel = df_plano_cliente_excel[['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'COD_GRP_CLIENTE', 'DESC_GRP_CLIENTE', 'PERIODO', 'METRICA', 'VALOR']]

# Padronizar os nomes das métricas
df_plano_cliente_excel['METRICA'] = df_plano_cliente_excel['METRICA'].replace({
    'Estatístico [KG]': 'ESTATISTICO_KG',
    'Orçamento [KG]': 'ORCAMENTO_KG',
    'Orçamento [R$]': 'ORCAMENTO_VAL',
    'Consenso [KG]': 'CONSENSO_KG',
    'Consenso [R$]': 'CONSENSO_VAL'
})

# Identificar períodos sem planejamento com base no total de CONSENSO_KG
periodos_excluir = df_plano_cliente_excel.loc[df_plano_cliente_excel['METRICA'].eq('CONSENSO_KG')].groupby('PERIODO')['VALOR'].sum()
periodos_excluir = periodos_excluir[periodos_excluir <= 0].index

# Excluir todas as linhas dos períodos sem planejamento
df_plano_cliente_excel = df_plano_cliente_excel.loc[~df_plano_cliente_excel['PERIODO'].isin(periodos_excluir)].copy()

# Garantir VALOR como numérico antes do pivot
df_plano_cliente_excel['VALOR'] = pd.to_numeric(df_plano_cliente_excel['VALOR'], errors='coerce').fillna(0)

# Transformar os valores da coluna METRICA em colunas
df_plano_cliente_excel = df_plano_cliente_excel.pivot_table(index=['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'COD_GRP_CLIENTE', 'DESC_GRP_CLIENTE', 'PERIODO'], columns='METRICA', values='VALOR', aggfunc='sum', fill_value=0).reset_index()

# Remover o nome do eixo criado pelo pivot
df_plano_cliente_excel.columns.name = None

# Criar colunas adicionais para produto e grupo de cliente
df_plano_cliente_excel[['COD_PROD', 'DESC_PROD']] = ''
df_plano_cliente_excel[['ORCAMENTO_KG', 'ORCAMENTO_VAL']] = 0

# Organizar as colunas na mesma estrutura do Parquet histórico
df_plano_cliente_excel = df_plano_cliente_excel[['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'COD_PROD', 'DESC_PROD', 'COD_GRP_CLIENTE', 'DESC_GRP_CLIENTE', 'PERIODO', 'ESTATISTICO_KG', 'ORCAMENTO_KG', 'ORCAMENTO_VAL', 'CONSENSO_KG', 'CONSENSO_VAL']]

# Criar coluna PLANO com o valor CLIENTE
df_plano_cliente_excel['PLANO'] = 'CLIENTE'

# Criar o CICLO com base no menor e maior PERIODO do plano
meses_abrev = {1: 'JAN', 2: 'FEV', 3: 'MAR', 4: 'ABR', 5: 'MAI', 6: 'JUN', 7: 'JUL', 8: 'AGO', 9: 'SET', 10: 'OUT', 11: 'NOV', 12: 'DEZ'}

periodo_min = df_plano_cliente_excel['PERIODO'].min()
periodo_max = df_plano_cliente_excel['PERIODO'].max()

ciclo = f"CICLO_{meses_abrev[periodo_min.month]}{str(periodo_min.year)[-2:]}_{meses_abrev[periodo_max.month]}{str(periodo_max.year)[-2:]}"
df_plano_cliente_excel['CICLO'] = ciclo

# Criar REVISAO com base na última revisão existente para o mesmo CICLO
df_bd_plano_painel_sop = pd.read_parquet(pasta_historico_planos / 'BD_PLANO_PAINEL_SOP.parquet')

if ciclo in df_bd_plano_painel_sop['CICLO'].values:
    revisoes_ciclo = df_bd_plano_painel_sop.loc[df_bd_plano_painel_sop['CICLO'].eq(ciclo), 'REVISAO']
    ultima_revisao_num = revisoes_ciclo.str.extract(r'(\d+)')[0].astype(int).max()
    nova_revisao = f'REV{ultima_revisao_num + 1}'
else:
    nova_revisao = 'REV1'

df_plano_cliente_excel['REVISAO'] = nova_revisao

# Criar coluna DATA_VERSAO com a data e hora atual
df_plano_cliente_excel['DATA_VERSAO'] = pd.Timestamp.now()

# Padronizar DATA_VERSAO para datetime64[ns]
df_plano_cliente_excel['DATA_VERSAO'] = df_plano_cliente_excel['DATA_VERSAO'].astype('datetime64[ns]')

# Padronizar as métricas numéricas como float
colunas_metricas = ['ESTATISTICO_KG', 'ORCAMENTO_KG', 'ORCAMENTO_VAL', 'CONSENSO_KG', 'CONSENSO_VAL']
df_plano_cliente_excel[colunas_metricas] = df_plano_cliente_excel[colunas_metricas].astype(float)

# Padronizar os códigos de produto e grupo de cliente como strings sem espaços em branco
df_plano_cliente_excel['COD_PROD'] = df_plano_cliente_excel['COD_PROD'].astype('string').str.strip()
df_plano_cliente_excel['COD_GRP_CLIENTE'] = df_plano_cliente_excel['COD_GRP_CLIENTE'].astype('string').str.strip()

# Excluir o arquivo temporário, se existir
if copia.exists():
    copia.unlink()

#### 04. CARREGANDO PLANO PRODUTO DO EXCEL

In [66]:
# -----------------------------------------------------------------------
# Carregar plano do painel em Excel do ciclo atual - PRODUTO
# -----------------------------------------------------------------------

# Definir arquivo original do painel e arquivo temporário
origem = pasta_painel / 'PREV_DEMANDA_KRONA.xlsb'
copia = pasta_painel / 'PREV_DEMANDA_KRONA_TEMP.xlsb'

# Excluir arquivo temporário anterior, caso exista
if copia.exists():
    copia.unlink()

# Criar uma cópia temporária do arquivo original
shutil.copy2(origem, copia)

# Carregar a aba do plano a partir da 4ª linha
df_plano_produto_excel = pd.read_excel(copia, sheet_name='PREV_COLAB_PRODUTO', engine='pyxlsb', skiprows=5)

# Definir os rótulos que devem ser mantidos
colunas_manter = ['REGIONAL', 'REGIONAL GESTOR', 'FAMILIA', 'COD_PROD', 'DESC_PROD',
 'ORÇAMENTO [KG]', 'ORÇAMENTO [R$]', 'ESTATÍSTICO [KG]', 'CONSENSO [KG]', 'CONSENSO [R$]']

# Ler e normalizar os rótulos da segunda linha do DataFrame
rotulos = df_plano_produto_excel.iloc[1].astype(str).str.replace('\n', ' ', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip().str.upper()

# Normalizar os nomes das colunas desejadas para comparação
colunas_manter_norm = [coluna.upper() for coluna in colunas_manter]

# Identificar as colunas físicas correspondentes aos rótulos desejados
colunas_selecionadas = [coluna for coluna, rotulo in zip(df_plano_produto_excel.columns, rotulos) if rotulo in colunas_manter_norm]

# Manter somente as colunas selecionadas
df_plano_produto_excel = df_plano_produto_excel[colunas_selecionadas]

# Eliminar linha 0 e reorganizar o índice
df_plano_produto_excel = df_plano_produto_excel.drop(index=0).reset_index(drop=True)

# Ler e normalizar os rótulos reais da primeira linha do DataFrame
rotulos = df_plano_produto_excel.iloc[0].astype(str).str.replace('\n', ' ', regex=False).str.replace(r'\s+', ' ', regex=True).str.strip()

# Identificar as colunas fixas
colunas_id = [coluna for coluna, rotulo in zip(df_plano_produto_excel.columns, rotulos) if rotulo in ['REGIONAL GESTOR', 'REGIONAL', 'FAMILIA', 'COD_PROD', 'DESC_PROD']]

# Identificar as colunas de período
colunas_periodo = [coluna for coluna in df_plano_produto_excel.columns if coluna not in colunas_id]

# Criar mapa entre coluna física e métrica correspondente
map_metrica = {coluna: rotulo for coluna, rotulo in zip(df_plano_produto_excel.columns, rotulos) if coluna in colunas_periodo}

# Remover a linha de rótulos antes da transposição
df_plano_produto_excel = df_plano_produto_excel.iloc[1:].reset_index(drop=True)

# Transpor as colunas mensais para linhas
df_plano_produto_excel = df_plano_produto_excel.melt(id_vars=colunas_id, value_vars=colunas_periodo, var_name='PERIODO', value_name='VALOR')

# Criar coluna com a métrica correspondente
df_plano_produto_excel['METRICA'] = df_plano_produto_excel['PERIODO'].map(map_metrica)

# Remover os sufixos .1, .2, .3 etc. gerados pelo pandas nas colunas de período
df_plano_produto_excel['PERIODO'] = df_plano_produto_excel['PERIODO'].astype(str).str.replace(r'\.\d+$', '', regex=True)

# Converter o período serial do Excel para data
df_plano_produto_excel['PERIODO'] = pd.to_datetime(pd.to_numeric(df_plano_produto_excel['PERIODO'], errors='coerce'), unit='D', origin='1899-12-30')

# Renomear as colunas fixas para o padrão final
map_colunas_id = {coluna: ('REGIONAL_GESTOR' if rotulo == 'REGIONAL GESTOR' else ('REGIONAL' if rotulo == 'REGIONAL' else rotulo)) for coluna, rotulo in zip(df_plano_produto_excel.columns[:len(colunas_id)], ['REGIONAL GESTOR', 'REGIONAL', 'FAMILIA', 'COD_PROD', 'DESC_PROD'])}
df_plano_produto_excel = df_plano_produto_excel.rename(columns=map_colunas_id)

# # Organizar as colunas finais
# Organizar as colunas finais
df_plano_produto_excel = df_plano_produto_excel[['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'COD_PROD', 'DESC_PROD', 'PERIODO', 'METRICA', 'VALOR']]

# Padronizar os nomes das métricas
df_plano_produto_excel['METRICA'] = df_plano_produto_excel['METRICA'].replace({
    'Estatístico [KG]': 'ESTATISTICO_KG',
    'Orçamento [KG]': 'ORCAMENTO_KG',
    'Orçamento [R$]': 'ORCAMENTO_VAL',
    'Consenso [KG]': 'CONSENSO_KG',
    'Consenso [R$]': 'CONSENSO_VAL'
})

# Identificar períodos sem planejamento com base no total de CONSENSO_KG
periodos_excluir = df_plano_produto_excel.loc[df_plano_produto_excel['METRICA'].eq('CONSENSO_KG')].groupby('PERIODO')['VALOR'].sum()
periodos_excluir = periodos_excluir[periodos_excluir <= 0].index

# Excluir todas as linhas dos períodos sem planejamento
df_plano_produto_excel = df_plano_produto_excel.loc[~df_plano_produto_excel['PERIODO'].isin(periodos_excluir)].copy()

# Garantir VALOR como numérico antes do pivot
df_plano_produto_excel['VALOR'] = pd.to_numeric(df_plano_produto_excel['VALOR'], errors='coerce').fillna(0)

# Transformar os valores da coluna METRICA em colunas
df_plano_produto_excel = df_plano_produto_excel.pivot_table(index=['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'COD_PROD', 'DESC_PROD', 'PERIODO'], columns='METRICA', values='VALOR', aggfunc='sum', fill_value=0).reset_index()

# Remover o nome do eixo criado pelo pivot
df_plano_produto_excel.columns.name = None

# Criar colunas adicionais para produto e grupo de cliente
df_plano_produto_excel[['COD_GRP_CLIENTE', 'DESC_GRP_CLIENTE']] = ''

# Organizar as colunas na mesma estrutura do Parquet histórico
df_plano_produto_excel = df_plano_produto_excel[['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'COD_PROD', 'DESC_PROD', 'COD_GRP_CLIENTE', 'DESC_GRP_CLIENTE', 'PERIODO', 'ESTATISTICO_KG', 'ORCAMENTO_KG', 'ORCAMENTO_VAL', 'CONSENSO_KG', 'CONSENSO_VAL']]

# Criar coluna PLANO com o valor PRODUTO
df_plano_produto_excel['PLANO'] = 'PRODUTO'

# Criar o CICLO com base no menor e maior PERIODO do plano
meses_abrev = {1: 'JAN', 2: 'FEV', 3: 'MAR', 4: 'ABR', 5: 'MAI', 6: 'JUN', 7: 'JUL', 8: 'AGO', 9: 'SET', 10: 'OUT', 11: 'NOV', 12: 'DEZ'}

periodo_min = df_plano_produto_excel['PERIODO'].min()
periodo_max = df_plano_produto_excel['PERIODO'].max()

ciclo = f"CICLO_{meses_abrev[periodo_min.month]}{str(periodo_min.year)[-2:]}_{meses_abrev[periodo_max.month]}{str(periodo_max.year)[-2:]}"
df_plano_produto_excel['CICLO'] = ciclo

# Criar REVISAO com base na última revisão existente para o mesmo CICLO
df_bd_plano_painel_sop = pd.read_parquet(pasta_historico_planos / 'BD_PLANO_PAINEL_SOP.parquet')

if ciclo in df_bd_plano_painel_sop['CICLO'].values:
    revisoes_ciclo = df_bd_plano_painel_sop.loc[df_bd_plano_painel_sop['CICLO'].eq(ciclo), 'REVISAO']
    ultima_revisao_num = revisoes_ciclo.str.extract(r'(\d+)')[0].astype(int).max()
    nova_revisao = f'REV{ultima_revisao_num + 1}'
else:
    nova_revisao = 'REV1'

df_plano_produto_excel['REVISAO'] = nova_revisao

# Criar coluna DATA_VERSAO com a data e hora atual
df_plano_produto_excel['DATA_VERSAO'] = pd.Timestamp.now()

# Padronizar DATA_VERSAO para datetime64[ns]
df_plano_produto_excel['DATA_VERSAO'] = df_plano_produto_excel['DATA_VERSAO'].astype('datetime64[ns]')

# Padronizar as métricas numéricas como float
colunas_metricas = ['ESTATISTICO_KG', 'ORCAMENTO_KG', 'ORCAMENTO_VAL', 'CONSENSO_KG', 'CONSENSO_VAL']
df_plano_produto_excel[colunas_metricas] = df_plano_produto_excel[colunas_metricas].astype(float)

# Padronizar os códigos de produto e grupo de cliente como strings sem espaços em branco
df_plano_produto_excel['COD_PROD'] = df_plano_produto_excel['COD_PROD'].astype('string').str.strip()
df_plano_produto_excel['COD_GRP_CLIENTE'] = df_plano_produto_excel['COD_GRP_CLIENTE'].astype('string').str.strip()

# Excluir o arquivo temporário, se existir
if copia.exists():
    copia.unlink()

#### 05. SALVANDO PLANOS TRATADOS DO EXCEL NO BANCO PARQUET

In [67]:
# Criar salvamento para planos regional, cliente, produto pois deverão ter a mesma DATA_VERSAO para quando necessitar buscar a ultima versão do plano pela DATA_VERSAO - salvando na BD_PLANO_PAINEL_SOP

# Criar a mesma DATA_VERSAO para todos os níveis do plano
data_versao = pd.Timestamp.now()

df_plano_produto_excel['DATA_VERSAO'] = data_versao
df_plano_cliente_excel['DATA_VERSAO'] = data_versao
df_plano_regional_excel['DATA_VERSAO'] = data_versao

# Unificar os planos da versão atual
df_plano_unificado = pd.concat([df_plano_produto_excel, df_plano_cliente_excel, df_plano_regional_excel], ignore_index=True)

# Caminho do banco histórico
arquivo_bd = pasta_historico_planos / 'BD_PLANO_PAINEL_SOP.parquet'

# Carregar histórico existente e adicionar nova versão
if arquivo_bd.exists():
    df_bd_plano_painel_sop = pd.read_parquet(arquivo_bd)
    df_bd_plano_painel_sop = pd.concat([df_bd_plano_painel_sop, df_plano_unificado], ignore_index=True)
else:
    df_bd_plano_painel_sop = df_plano_unificado.copy()

# Salvar banco completo
df_bd_plano_painel_sop.to_parquet(arquivo_bd, index=False)

#### 06. CARREGAR ULTIMO PLANO REGIONAL SALVO NO PARQUET E DESAGREGAR POR EMPRESA/PRODUTO

In [ ]:
# Carregar o banco histórico do plano S&OP
df_bd_plano_painel_sop = pd.read_parquet(pasta_historico_planos / 'BD_PLANO_PAINEL_SOP.parquet')

# Identificar o registro com a DATA_VERSAO mais recente
registro_ultima_versao = df_bd_plano_painel_sop.loc[df_bd_plano_painel_sop['DATA_VERSAO'].idxmax()]

# Identificar o CICLO e a REVISAO correspondentes à última DATA_VERSAO
ultimo_ciclo = registro_ultima_versao['CICLO']
ultima_revisao = registro_ultima_versao['REVISAO']

# Filtrar somente o plano REGIONAL pertencente ao último CICLO e à última REVISAO
df_ultimo_plano_regional = df_bd_plano_painel_sop.loc[df_bd_plano_painel_sop['CICLO'].eq(ultimo_ciclo) & df_bd_plano_painel_sop['REVISAO'].eq(ultima_revisao) & df_bd_plano_painel_sop['PLANO'].eq('REGIONAL')].copy()

# Definir as colunas utilizadas na agregação
colunas_agrupamento = ['REGIONAL_GESTOR', 'REGIONAL', 'FAMILIA', 'PERIODO', 'CICLO']

# Agregar o último plano pelas dimensões principais
df_ultimo_plano_regional = df_ultimo_plano_regional.groupby(colunas_agrupamento, as_index=False)[['CONSENSO_KG']].sum()

# Definir quantidade de períodos utilizados na desagregação
ult_meses_hist_vend = 12

# Carregar o histórico de vendas utilizado para calcular a desagregação
df_vendas_krona = pd.read_parquet(pasta_staging_parquet / 'df_vendas_krona.parquet')

# Identificar os 12 últimos períodos disponíveis na base
ultimos_periodos = df_vendas_krona['PERIODO'].dropna().drop_duplicates().nlargest(ult_meses_hist_vend)

# Filtrar somente os 12 últimos períodos
df_vendas_krona = df_vendas_krona.loc[df_vendas_krona['PERIODO'].isin(ultimos_periodos)].copy()

# -----------------------------------------------------------------------
# Preparar histórico para desagregação por EMPRESA + PRODUTO
# -----------------------------------------------------------------------

# Definir as chaves do histórico
chaves_historico = ['EMPRESA', 'COD_PROD', 'REGIONAL', 'FAMILIA']

# Agregar o volume histórico dos últimos 12 períodos
df_hist_desag = df_vendas_krona.groupby(chaves_historico, as_index=False)['VOL_VENDA'].sum()

# Calcular o volume histórico total dentro de cada REGIONAL + FAMILIA
df_hist_desag['TOTAL_HIST_FAMILIA'] = df_hist_desag.groupby(['REGIONAL', 'FAMILIA'])['VOL_VENDA'].transform('sum')

# Calcular a participação histórica de cada combinação EMPRESA + PRODUTO dentro da REGIONAL + FAMILIA
df_hist_desag['PARTIC_HIST'] = np.where(df_hist_desag['TOTAL_HIST_FAMILIA'] > 0, df_hist_desag['VOL_VENDA'] / df_hist_desag['TOTAL_HIST_FAMILIA'], 0)

# Associar a participação histórica de EMPRESA + PRODUTO ao plano regional por REGIONAL + FAMILIA
df_plano_desag_emp_prod = df_ultimo_plano_regional.merge(
    df_hist_desag[['EMPRESA', 'COD_PROD', 'REGIONAL', 'FAMILIA', 'PARTIC_HIST']],
    on=['REGIONAL', 'FAMILIA'],
    how='left'
)

# Calcular o consenso desagregado respeitando o volume total planejado da família
df_plano_desag_emp_prod['CONSENSO_KG_DESAG'] = df_plano_desag_emp_prod['CONSENSO_KG'] * df_plano_desag_emp_prod['PARTIC_HIST']

# Excluir algumas colunas para buscar da DIM_PRODUTOS atualizado
df_plano_desag_emp_prod = df_plano_desag_emp_prod.drop(columns=['PARTIC_HIST', 'FAMILIA', 'LINHA'], errors='ignore')

# Carregar a dimensão de produtos
df_dim_produtos = pd.read_parquet(pasta_staging_parquet / 'DIM_PRODUTOS_KRONA.parquet')

# Preparar dimensão de produtos para atualização por COD_PROD
df_dim_produtos = df_dim_produtos.drop_duplicates(subset=['COD_PROD']).set_index('COD_PROD')  

# Buscar informações atualizadas de DESC_PROD, PESO_UNIT, FAMILIA e LINHA a partir da dimensão de produtos, pelo COD_PROD
df_plano_desag_emp_prod['DESC_PROD'] = df_plano_desag_emp_prod['COD_PROD'].map(df_dim_produtos['DESC_PROD'])
df_plano_desag_emp_prod['PESO_UNIT'] = df_plano_desag_emp_prod['COD_PROD'].map(df_dim_produtos['PESO_UNIT'])
df_plano_desag_emp_prod['FAMILIA'] = df_plano_desag_emp_prod['COD_PROD'].map(df_dim_produtos['FAMILIA'])
df_plano_desag_emp_prod['LINHA'] = df_plano_desag_emp_prod['COD_PROD'].map(df_dim_produtos['LINHA'])

# Calcular PREV_PCS arredondando sempre para cima
df_plano_desag_emp_prod['PREV_PCS'] = np.where(
    df_plano_desag_emp_prod['PESO_UNIT'] > 0,
    np.ceil(df_plano_desag_emp_prod['CONSENSO_KG_DESAG'] / df_plano_desag_emp_prod['PESO_UNIT']),
    0
)

# Excluir Colunas desnecessárias
df_plano_desag_emp_prod = df_plano_desag_emp_prod.drop(columns=['PESO_UNIT', 'CONSENSO_KG'], errors='ignore')

# Renomear coluna DESC_PROD para DESC_PRODUTO
df_plano_desag_emp_prod = df_plano_desag_emp_prod.rename(columns={'CONSENSO_KG_DESAG': 'PREV_KG'})

# Criar coluna PLANO com o valor REGIONAL
df_plano_desag_emp_prod['PLANO'] = 'REGIONAL'

# Adicionar a revisão correspondente ao plano desagregado
df_plano_desag_emp_prod['REVISAO'] = nova_revisao

# Criar coluna TIPO = PREV_CONSENSO
df_plano_desag_emp_prod['TIPO'] = 'PREV_CONSENSO'

# Criar coluna DATA_VERSAO
df_plano_desag_emp_prod['DATA_VERSAO'] = df_bd_plano_painel_sop['DATA_VERSAO'].max().date()

# Organizar Colunas
df_plano_desag_emp_prod = df_plano_desag_emp_prod[['EMPRESA', 'COD_PROD', 'DESC_PROD', 'FAMILIA', 'LINHA', 'REGIONAL', 'REGIONAL_GESTOR', 'PLANO', 'PERIODO', 'CICLO', 'REVISAO', 'TIPO', 'PREV_KG', 'PREV_PCS', 'DATA_VERSAO']]

#### 07. CARREGAR ULTIMO PLANO CLIENTE SALVO NO PARQUET E DESAGREGAR

#### 08. CARREGAR ULTIMO PLANO PRODUTO SALVO NO PARQUET E DESAGREGAR

#### 09. CARREGAR E PREPARAR PLANO DE LANÇAMENTO DE PRODUTOS DO CICLO

In [ ]:
# Criar plano final do lançamento de produtos
df_demanda_produtos_lancamento = pd.read_parquet(pasta_staging_parquet / 'df_demanda_produtos_lancamento.parquet')

# Salvando plano no formato original captado no inicio do ETL
df_demanda_produtos_lancamento.to_excel(pasta_saida_plan_producao / f'plano_lancamento_{ultimo_ciclo}.xlsx', index=False)

# Criar mapping de CD para EMPRESA
map_cd_empresa = {
    'CD: CO': 'KRONA - AP GOIÂNIA',
    'CD: VQ': 'VIQUA',
    'CD: TM': 'TOPMAX',
    'CD: NE': 'KRONA - NORDESTE',
    'CD: MT': 'KRONA - MATRIZ'
}

# Criar plano final do lançamento de produtos
df_demanda_produtos_lancamento = pd.read_parquet(
    pasta_staging_parquet / 'df_demanda_produtos_lancamento.parquet'
)

# Renomear as colunas de CD conforme o mapping de EMPRESA
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento.rename(
    columns=map_cd_empresa
)

# Multiplicar a participação de cada EMPRESA pela quantidade total do lançamento
colunas_empresa = list(map_cd_empresa.values())
df_demanda_produtos_lancamento[colunas_empresa] = df_demanda_produtos_lancamento[colunas_empresa].mul(
    df_demanda_produtos_lancamento['QTD_LANC'],
    axis=0
)

# Identificar as colunas de EMPRESA cuja soma é zero
colunas_empresa_zero = [
    coluna for coluna in colunas_empresa
    if df_demanda_produtos_lancamento[coluna].sum() == 0
]

# Eliminar as colunas de EMPRESA sem volume
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento.drop(columns=colunas_empresa_zero)

# Eliminar colunas desnecessárias
colunas_desnecessarias = ['MARCA', 'PROCESSO', 'FAMILIA_SOP', 'VOL_LANC', 'QTD_LANC', 'FAMILIA', 'LINHA']
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento.drop(columns=colunas_desnecessarias)

# Identificar as colunas de EMPRESA que ainda existem no DataFrame
colunas_empresa = [coluna for coluna in map_cd_empresa.values() if coluna in df_demanda_produtos_lancamento.columns]

# Transpor as colunas de EMPRESA para linhas
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento.melt(
    id_vars=['COD_PROD', 'DESC_PROD', 'PERIODO'],
    value_vars=colunas_empresa,
    var_name='EMPRESA',
    value_name='QTD_LANC'
)

# Arredondar QTD_LANC sempre para cima
df_demanda_produtos_lancamento['QTD_LANC'] = np.ceil(df_demanda_produtos_lancamento['QTD_LANC']).astype(int)

# Eliminar linhas com QTD_LANC igual a zero
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento[df_demanda_produtos_lancamento['QTD_LANC'] != 0]

# Buscar dados por COD_PROD na dim_produtos
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento.merge(
    df_dim_produtos[['FAMILIA', 'LINHA', 'PESO_UNIT']],
    how='left',
    on='COD_PROD'
)

# Calcular KG_LANC
df_demanda_produtos_lancamento['KG_LANC'] = df_demanda_produtos_lancamento['QTD_LANC'] * df_demanda_produtos_lancamento['PESO_UNIT']

# Eliminar colunas desnecessárias após o cálculo de KG_LANC
colunas_desnecessarias_apos_kg = ['PESO_UNIT']
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento.drop(columns=colunas_desnecessarias_apos_kg)

# COLUNA TIPO = LANCAMENTO_PRODUTO
df_demanda_produtos_lancamento['TIPO'] = 'LANC_PRODUTOS'

# Criar colunas REGIONAL, REGIONAL GESTOR Vazias
df_demanda_produtos_lancamento['REGIONAL'] = ''
df_demanda_produtos_lancamento['REGIONAL_GESTOR'] = ''

# Criar Coluna PLANO = LANC_PRODUTOS
df_demanda_produtos_lancamento['PLANO'] = 'LANC_PRODUTOS'

# Criar coluna Ciclo = ultimo_ciclo
df_demanda_produtos_lancamento['CICLO'] = ultimo_ciclo

# Criar coluna REVISAO
df_demanda_produtos_lancamento['REVISAO'] = ultima_revisao

# Criar coluna DATA_VERSAO
df_demanda_produtos_lancamento['DATA_VERSAO'] = df_bd_plano_painel_sop['DATA_VERSAO'].max().date()

# Renomear colunas QTD_LANC e KG_LANC
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento.rename(columns={
    'QTD_LANC': 'PREV_PCS',
    'KG_LANC': 'PREV_KG'
})

# Organizar Colunas
df_demanda_produtos_lancamento = df_demanda_produtos_lancamento[['EMPRESA', 'COD_PROD', 'DESC_PROD', 'FAMILIA', 'LINHA', 'REGIONAL', 'REGIONAL_GESTOR', 'PLANO', 'PERIODO', 'CICLO', 'REVISAO', 'TIPO', 'PREV_KG', 'PREV_PCS', 'DATA_VERSAO']]

#### 10. SALVAR OS PLANOS DESAGREGADOS REGIONAL, PRODUTO, CLIENTE, GERANDO DATA_VERSAO PARA TODOS

In [37]:
# Carregar o histórico do plano desagregado, caso exista
if (pasta_historico_planos / 'BD_PLANO_DESAG_SOP.parquet').exists():
    df_bd_plano_desag_sop = pd.read_parquet(pasta_historico_planos / 'BD_PLANO_DESAG_SOP.parquet')
    df_bd_plano_desag_sop = pd.concat([df_bd_plano_desag_sop, df_plano_desag_emp_prod, df_demanda_produtos_lancamento], ignore_index=True)
else:
    df_bd_plano_desag_sop = pd.concat([df_plano_desag_emp_prod, df_demanda_produtos_lancamento], ignore_index=True)

# Salvar o DataFrame atualizado no arquivo Parquet
df_bd_plano_desag_sop.to_parquet(pasta_historico_planos / 'BD_PLANO_DESAG_SOP.parquet', index=False)

#### 10. GERAR ARQUIVO ENVIADO AO PLANEJAMENTO/PLANO PRODUCAO

In [58]:
# Carregar o banco histórico do plano S&OP
df_bd_plano_desag_sop = pd.read_parquet(pasta_historico_planos / 'BD_PLANO_DESAG_SOP.parquet')

# Identificar o registro com a DATA_VERSAO mais recente
registro_ultima_versao = df_bd_plano_desag_sop.loc[df_bd_plano_desag_sop['DATA_VERSAO'].idxmax()]

# Identificar o CICLO e a REVISAO correspondentes à última DATA_VERSAO
ultimo_ciclo = registro_ultima_versao['CICLO']
ultima_revisao = registro_ultima_versao['REVISAO']

# Filtrar somente o último CICLO e à última REVISAO
df_bd_plano_desag_sop = df_bd_plano_desag_sop.loc[df_bd_plano_desag_sop['CICLO'].eq(ultimo_ciclo) & df_bd_plano_desag_sop['REVISAO'].eq(ultima_revisao)].copy()

# Criar plano final da desagregação para gerar arquivo final em excel para analise de capacidade
df_plano_PREVISAO = df_bd_plano_desag_sop.copy()

# Agrupar valores
colunas_grupo = ['EMPRESA', 'COD_PROD', 'DESC_PROD', 'FAMILIA', 'LINHA', 'PERIODO', 'REVISAO', 'TIPO']
colunas_valor = ['PREV_PCS', 'PREV_KG']
df_plano_PREVISAO = df_plano_PREVISAO.groupby(colunas_grupo, as_index=False)[colunas_valor].sum()

# Converter PERIODO para data sem horário
df_plano_PREVISAO['PERIODO'] = pd.to_datetime(df_plano_PREVISAO['PERIODO']).dt.date

# Eliminar arquivos anteriores de demanda do plano de produção
for arquivo in pasta_saida_plan_producao.glob('demanda_plano_producao_*.xlsx'):
    arquivo.unlink()
    
# Salvar novo arquivo Excel
df_plano_PREVISAO.to_excel(pasta_saida_plan_producao / f'demanda_plano_producao_{ultimo_ciclo}.xlsx', index=False)

print(f"✅ 'demanda_plano_producao_{ultimo_ciclo}.xlsx' gerado com sucesso!")

✅ 'demanda_plano_producao_CICLO_OUT26_JAN27.xlsx' gerado com sucesso!


#### 11. GERAR ARQUIVO ENVIADO AO COMERCIAL PARA VALIDAÇÃO DO PLANO

In [59]:
# Gerar arquivo de saída para validação do Analista de SOP

# Carregar o banco histórico do plano S&OP
df_bd_plano_desag_sop = pd.read_parquet(pasta_historico_planos / 'BD_PLANO_DESAG_SOP.parquet')

# Identificar o registro com a DATA_VERSAO mais recente
registro_ultima_versao = df_bd_plano_desag_sop.loc[df_bd_plano_desag_sop['DATA_VERSAO'].idxmax()]

# Identificar o CICLO e a REVISAO correspondentes à última DATA_VERSAO
ultimo_ciclo = registro_ultima_versao['CICLO']
ultima_revisao = registro_ultima_versao['REVISAO']

# Filtrar somente o último CICLO e à última REVISAO
df_plano_saida_comercial_consenso = df_bd_plano_desag_sop.loc[df_bd_plano_desag_sop['CICLO'].eq(ultimo_ciclo) & df_bd_plano_desag_sop['REVISAO'].eq(ultima_revisao)].copy()

# Agrupar dados
colunas_grupo = ['COD_PROD', 'DESC_PROD', 'FAMILIA', 'LINHA', 'REGIONAL', 'REGIONAL_GESTOR', 'PERIODO', 'CICLO', 'TIPO']
colunas_valor = ['PREV_PCS', 'PREV_KG']
df_plano_saida_comercial_consenso = df_plano_saida_comercial_consenso.groupby(colunas_grupo).agg({col: 'sum' for col in colunas_valor}).reset_index()

# Eliminar arquivos anteriores de demanda do plano de produção
for arquivo in pasta_saida_plan_producao.glob('plano_saida_estatistico_consenso_*.xlsx'):
    arquivo.unlink()

df_plano_saida_comercial_consenso.to_excel(pasta_saida_plan_producao / f'plano_saida_comercial_{ultimo_ciclo}.xlsx', index=False)

In [ ]:
timer.finalizar()
print("🎯 Processo concluído com sucesso!")

In [ ]:
# Após desenvolver transformação de cada tipo (neste momento falta produto), criar o salvamento de todos eles, que ainda não foi criado e devem ser salvos juntos, para inserir mesma data e hora da versão
# Analise e pensar se a desagregação deveria ser em cada um (PRODUTO, CLIENTE, REGIONAL), ou deveria ser juntos somando os dados dos 3. Lembrando, que a demanda por produto deveria entrar diretamente, sem precisar desagregar
# Ao Terminar o DEV, limpar a REV2 da base
# PAssar pelo gpt as bases, e os scripts, e ver quais apagar



# Onde parei: Criamos as transformações para todos os 3 tipos de plano, agora é avançar para o salvamento dos planos no AGREG
# IMPORTANTE: RETIRAR A DATA DA VERSÃO DOS PLANOS E COLOCAR NO SALVAMENTO DELES CONFORME GPT - Montei mas precisa testar, coloca PLANO incorreto nas ETLS






In [100]:
# No final:
# 1. Excluir dados do arquivo 04_HISTORICO_PLANO/BD_PLANO_PAINEL_SOP, Mantendo apenas REV1
# MANTER somente REV1 no Parquet e salvar
df_bd_plano_painel_sop = df_bd_plano_painel_sop[df_bd_plano_painel_sop['REVISAO'] == 'REV1']
df_bd_plano_painel_sop.to_parquet(pasta_historico_planos / 'BD_PLANO_PAINEL_SOP.parquet', index=False)